<a href="https://colab.research.google.com/github/HLZHarry/Alpha-Lens-TPM/blob/main/03_AI_Macro_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In this phase, we are solving a classic institutional problem: Market data is "Lagged" (Backward-looking), but Central Bank text is "Forward-looking."

We will build an agent that reads a simulated "Federal Reserve Statement" and converts the "Hawkish" (High Interest Rate) or "Dovish" (Low Interest Rate) tone into a numerical Regime Score.

Enhanced with RAG solution:
1. The Problem: FOMC Statements are short. The real "Alpha" is often hidden in the Minutes (released 3 weeks later) or in how the tone has changed compared to the previous meeting.
2. The RAG Solution: We will store all historical Statements and Minutes in a Vector Database (ChromaDB). When we score a date, the system "retrieves" the most recent documents to give the AI context.

In [1]:
!pip install -U openai chromadb

In [2]:
import requests
from bs4 import BeautifulSoup
from openai import OpenAI
import chromadb
from chromadb.utils import embedding_functions
import pandas as pd
import numpy as np
import time
import os
from google.colab import userdata, drive

In [3]:
# 1. INITIALIZE & MOUNT DRIVE
drive.mount('/content/drive')
path = "/content/drive/MyDrive/Alpha-Lens-Project"
os.makedirs(f"{path}/chroma_v3", exist_ok=True)

Mounted at /content/drive


In [4]:
# 2. SETUP THE STACK
# Using FinBERT for specialized financial understanding
finbert_ef = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="ProsusAI/finbert")
client = OpenAI(api_key=userdata.get('OPENAI_API_KEY'))

# PERSISTENT ChromaDB: Saves the database to your Drive
chroma_client = chromadb.PersistentClient(path=f"{path}/chroma_v3")
collection = chroma_client.get_or_create_collection(
    name="fomc_intelligence",
    embedding_function=finbert_ef
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
classifier.bias              | UNEXPECTED |  | 
bert.embeddings.position_ids | UNEXPECTED |  | 
classifier.weight            | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
# 3. COMPLETE MEETING DATES (2021-2026)
meeting_dates = [
    '20210127', '20210317', '20210428', '20210616', '20210728', '20210922', '20211103', '20211215',
    '20220126', '20220316', '20220504', '20220615', '20220727', '20220921', '20221102', '20221214',
    '20230201', '20230322', '20230503', '20230614', '20230726', '20230920', '20231101', '20231213',
    '20240131', '20240320', '20240501', '20240612', '20240731', '20240918', '20241107', '20241218',
    '20250129', '20250319', '20250507', '20250618', '20250730', '20250917', '20251029', '20251210',
    '20260128'
]

In [6]:
# 4. RESILIENT SCRAPER
def scrape_fed_docs(date):
    headers = {'User-Agent': 'Mozilla/5.0'}
    combined_docs = []
    # Scraping Statement + Minutes
    urls = {
        "S": f"https://www.federalreserve.gov/newsevents/pressreleases/monetary{date}a.htm",
        "M": f"https://www.federalreserve.gov/monetarypolicy/fomcminutes{date}.htm"
    }
    for key, url in urls.items():
        try:
            r = requests.get(url, headers=headers, timeout=10)
            if r.status_code == 200:
                soup = BeautifulSoup(r.content, 'html.parser')
                text = " ".join([p.text for p in soup.find_all('p') if len(p.text) > 100])
                if len(text) > 200:
                    combined_docs.append(text)
        except: continue
    return "\n---\n".join(combined_docs)

In [7]:
# 5. EXECUTION & SCORING LOOP
print("🚀 Starting Production RAG Pipeline...")
results = []

for date in meeting_dates:
    print(f"Processing {date}...", end=" ")

    # A. Scrape Text
    full_text = scrape_fed_docs(date)
    if not full_text or len(full_text) < 100:
        print("❌ Scrape Failed (Empty)")
        results.append({'Date': pd.to_datetime(date), 'AI_Sentiment': 0.0})
        continue

    # B. Store in Vector DB (Upsert)
    collection.upsert(
        documents=[full_text[:4000]],
        metadatas=[{"date": date}],
        ids=[f"id_{date}"]
    )

    # C. RAG STEP: Retrieve context from the Database
    # This retrieves the most similar previous meetings to compare tone
    history = collection.query(query_texts=["inflation, interest rates, and employment outlook"], n_results=2)
    context_text = " ".join(history['documents'][0])

    # D. ANALYZE: GPT-4o with Context
    try:
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=[
                {"role": "system", "content": "You are a Senior TPM Analyst. Compare the current text to historical context for policy shifts."},
                {"role": "user", "content": f"HISTORICAL CONTEXT:\n{context_text[:1500]}\n\nCURRENT STATEMENT:\n{full_text[:2000]}\n\nTASK: Score the sentiment from -1.0 (Hawkish) to 1.0 (Dovish). Return ONLY the float."}
            ],
            temperature=0
        )
        score = float(response.choices[0].message.content.strip())
        results.append({'Date': pd.to_datetime(date), 'AI_Sentiment': score})
        print(f"✅ Score: {score}")
    except Exception as e:
        print(f"⚠️ Error: {e}")
        results.append({'Date': pd.to_datetime(date), 'AI_Sentiment': 0.0})

    time.sleep(1) # Throttling

🚀 Starting Production RAG Pipeline...
Processing 20210127... ✅ Score: 0.8
Processing 20210317... ✅ Score: 0.8
Processing 20210428... ✅ Score: 0.5
Processing 20210616... ✅ Score: 0.5
Processing 20210728... ✅ Score: 0.5
Processing 20210922... ✅ Score: 0.5
Processing 20211103... ✅ Score: 0.5
Processing 20211215... ✅ Score: 0.5
Processing 20220126... ✅ Score: -0.3
Processing 20220316... ✅ Score: -0.5
Processing 20220504... ✅ Score: -0.5
Processing 20220615... ✅ Score: -0.7
Processing 20220727... ✅ Score: -0.7
Processing 20220921... ✅ Score: -0.7
Processing 20221102... ✅ Score: -0.5
Processing 20221214... ✅ Score: -0.7
Processing 20230201... ✅ Score: -0.5
Processing 20230322... ✅ Score: -0.5
Processing 20230503... ✅ Score: -0.5
Processing 20230614... ✅ Score: 0.2
Processing 20230726... ✅ Score: -0.5
Processing 20230920... ✅ Score: -0.3
Processing 20231101... ✅ Score: -0.3
Processing 20231213... ✅ Score: -0.3
Processing 20240131... ✅ Score: -0.3
Processing 20240320... ✅ Score: -0.3
Processin

In [8]:
# 6. INTEGRATE & SAVE
sentiment_df = pd.DataFrame(results)
df_features = pd.read_csv(f"{path}/engineered_features_v2.csv")
df_features['Date'] = pd.to_datetime(df_features['Date'])

df_final = pd.merge(df_features, sentiment_df, on='Date', how='left')
# Propagation: The Fed's mood lasts until the next meeting
df_final['AI_Sentiment'] = df_final.groupby('Ticker')['AI_Sentiment'].ffill().fillna(0)

df_final.to_csv(f"{path}/final_hybrid_data_v3.csv", index=False)
print("\n✅ Phase 3 Complete. Final dataset saved as 'final_hybrid_data_v3.csv'")


✅ Phase 3 Complete. Final dataset saved as 'final_hybrid_data_v3.csv'
